<a href="https://colab.research.google.com/github/munnurumahesh03-coder/kaggle-predicting-loan-payback/blob/main/03_Base_Models_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Imports and Setup ---

# Core Libraries
import pandas as pd
import numpy as np
import os
import gc
import warnings

# Scikit-learn Preprocessing and Metrics
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# Hyperparameter Tuning
import optuna

# --- Configuration ---

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore",module="lightgbm")

# Set a seed for reproducibility
SEED = 42

# Define the number of folds for cross-validation
N_SPLITS = 5

# Define the number of Optuna trials for each model type
# As per our strategy: 20 for complex models, 20 for simpler ones
N_TRIALS_BOOSTING = 30
N_TRIALS_SIMPLE = 20

# --- Display Settings ---
# Configure pandas to display all columns
pd.set_option('display.max_columns', None)

print("Cell 1: All libraries imported and configurations set.")
print(f"Optuna version: {optuna.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"CatBoost version: {cb.__version__}")

Cell 1: All libraries imported and configurations set.
Optuna version: 4.5.0
LightGBM version: 4.6.0
XGBoost version: 2.0.3
CatBoost version: 1.2.8


In [ ]:
# --- Load and Prepare Data (Corrected) ---

# Define file paths based on your provided input directory
TRAIN_FEAT_PATH = '/kaggle/input/02-feature-engineering-ipynb/train_featured_v2.csv'
TEST_FEAT_PATH = '/kaggle/input/02-feature-engineering-ipynb/test_featured_v2.csv'

# Load the datasets
print("Loading datasets from Notebook 2...")
try:
    train_df = pd.read_csv(TRAIN_FEAT_PATH)
    test_df = pd.read_csv(TEST_FEAT_PATH)
    print(f"Train data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please double-check that the input directory is '/kaggle/input/02-feature-engineering-ipynb/'.")
    # Stop execution if files are not found
    raise

# --- Feature, Target, and ID Separation ---

# Define the target column name
TARGET = 'loan_paid_back'

# Separate features (X) and target (y) from the training data
# Corrected: 'id' is not in the training data, so no need to drop it.
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]

# Separate the submission IDs and features from the test data
# Corrected: We save the test IDs for the final submission file.
test_ids = test_df['id']
X_test = test_df.drop(columns=['id'])

# --- Identify Feature Types ---

# Identify categorical features by their data type
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Identify numerical features
numerical_features = X.select_dtypes(include=np.number).columns.tolist()

# Display the findings
print(f"\nTarget column: '{TARGET}'")
print(f"Number of features: {X.shape[1]}")
print(f"Number of categorical features: {len(categorical_features)}")
print(f"Categorical features: {categorical_features}")
print(f"Number of numerical features: {len(numerical_features)}")
print(f"Test IDs shape: {test_ids.shape}")


# --- Memory Management ---
# Explanation: We delete the original large dataframes and run the garbage collector
# to free up RAM, preventing memory errors during model training.
del train_df, test_df
gc.collect()

print("\nCell 2: Data loaded and prepared successfully.")

Loading datasets from Notebook 2...
Train data shape: (593994, 24)
Test data shape: (254569, 24)

Target column: 'loan_paid_back'
Number of features: 23
Number of categorical features: 6
Categorical features: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
Number of numerical features: 17
Test IDs shape: (254569,)

Cell 2: Data loaded and prepared successfully.


# **Automated Hyperparameter Tuning**

In [ ]:
# --- The Complete ModelTuner Class (Final Version) ---

class ModelTuner:
    def __init__(self, numerical_features, categorical_features, n_splits, seed):
        self.n_splits = n_splits
        self.seed = seed

        # Preprocessor for models that need it (LogReg, LGBM-RF, XGBoost)
        self.preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), numerical_features),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
            ],
            remainder='passthrough'
        )

        # --- ALL MODELS ARE DEFINED HERE ---
        self.models = {
            'LogisticRegression': LogisticRegression(class_weight='balanced', random_state=self.seed, solver='liblinear'),
            'LGBM-RF': lgb.LGBMClassifier(boosting_type='rf', bagging_freq=1, device='gpu', random_state=self.seed, n_jobs=-1, verbosity=-1),
            'LightGBM': lgb.LGBMClassifier(device='gpu', random_state=self.seed, n_jobs=-1, verbosity=-1),
            'XGBoost': xgb.XGBClassifier(device='cuda', random_state=self.seed),
            'CatBoost': cb.CatBoostClassifier(task_type='GPU', random_seed=self.seed, verbose=0)
        }

        # --- ALL PARAMETER SPACES ARE DEFINED HERE ---
        self.params_search_space = {
            'LogisticRegression': lambda trial: {'classifier__C': trial.suggest_float('C', 1e-4, 1e2, log=True)},
            'LGBM-RF': lambda trial: {'classifier__n_estimators': trial.suggest_int('n_estimators', 100, 1000), 'classifier__max_depth': trial.suggest_int('max_depth', 5, 20), 'classifier__bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.99), 'classifier__feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.99), 'classifier__verbosity': -1},
            # Note: No 'classifier__' prefix for LightGBM and CatBoost params
            'LightGBM': lambda trial: {'n_estimators': trial.suggest_int('n_estimators', 500, 2000), 'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True), 'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 10), 'num_leaves': trial.suggest_int('num_leaves', 20, 300), 'verbosity': -1},
            'XGBoost': lambda trial: {'classifier__n_estimators': trial.suggest_int('n_estimators', 500, 2000), 'classifier__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True), 'classifier__scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 10)},
            'CatBoost': lambda trial: {'iterations': trial.suggest_int('iterations', 500, 2000), 'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True), 'auto_class_weights': trial.suggest_categorical('auto_class_weights', ['Balanced', 'SqrtBalanced'])}
        }

    def objective(self, trial, model_name, X_data, y_data):
        model = self.models[model_name]
        params = self.params_search_space[model_name](trial)
        cv_scores = []
        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.seed)

        # Special path for models that handle categories natively
        if model_name in ['LightGBM', 'CatBoost']:
            model.set_params(**params)
            for train_idx, val_idx in skf.split(X_data, y_data):
                X_train, X_val = X_data.iloc[train_idx], X_data.iloc[val_idx]
                y_train, y_val = y_data.iloc[train_idx], y_data.iloc[val_idx]

                # Use early stopping for efficiency
                model.fit(X_train, y_train,
                          eval_set=[(X_val, y_val)],
                          callbacks=[lgb.early_stopping(50, verbose=False)])

                preds = model.predict_proba(X_val)[:, 1]
                cv_scores.append(roc_auc_score(y_val, preds))
        else: # Standard path for models that need preprocessing
            pipeline = Pipeline(steps=[('preprocessor', self.preprocessor), ('classifier', model)])
            pipeline.set_params(**params)
            for train_idx, val_idx in skf.split(X_data, y_data):
                X_train, X_val = X_data.iloc[train_idx], X_data.iloc[val_idx]
                y_train, y_val = y_data.iloc[train_idx], y_data.iloc[val_idx]
                pipeline.fit(X_train, y_train)
                preds = pipeline.predict_proba(X_val)[:, 1]
                cv_scores.append(roc_auc_score(y_val, preds))

        return np.mean(cv_scores)

    def tune(self, model_name, n_trials, X_data, y_data):
        study = optuna.create_study(direction='maximize')
        study.optimize(lambda trial: self.objective(trial, model_name, X_data, y_data), n_trials=n_trials)
        return study.best_params, study.best_value

print("Cell 3: Complete ModelTuner class defined successfully.")


Cell 3: Complete ModelTuner class defined successfully.


In [ ]:
# --- Logistic Regression - Tuning and Prediction ---

MODEL_NAME = 'LogisticRegression'
print(f"--- Starting: {MODEL_NAME} ---")

# Initialize result dictionaries if they don't exist
if 'best_params_all' not in locals():
    best_params_all = {}
    oof_preds_all = {}
    test_preds_all = {}

# 1. Tune the model
tuner = ModelTuner(numerical_features, categorical_features, n_splits=N_SPLITS, seed=SEED)
n_trials = N_TRIALS_SIMPLE

print(f"Step 1: Tuning with {n_trials} trials...")
best_params, best_value = tuner.tune(MODEL_NAME, n_trials, X, y) # Using the original X, y
best_params_all[MODEL_NAME] = best_params
print(f"   ✅ Best Score: {best_value:.6f}")
print(f"   ✅ Best Params: {best_params}")

# 2. Generate predictions
print(f"\nStep 2: Generating predictions...")
model = tuner.models[MODEL_NAME]
final_params = {k.replace('classifier__', ''): v for k, v in best_params.items()}
model.set_params(**final_params)

pipeline = Pipeline(steps=[('preprocessor', tuner.preprocessor), ('classifier', model)])
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    pipeline.fit(X_train, y_train)
    oof_preds[val_idx] = pipeline.predict_proba(X_val)[:, 1]
    test_preds += pipeline.predict_proba(X_test)[:, 1] / N_SPLITS

oof_preds_all[MODEL_NAME] = oof_preds
test_preds_all[MODEL_NAME] = test_preds
final_oof_score = roc_auc_score(y, oof_preds)
print(f"\n   ✅ Final OOF ROC AUC for {MODEL_NAME}: {final_oof_score:.6f}")
print(f"--- Completed: {MODEL_NAME} ---")
gc.collect()

[I 2025-11-18 17:04:45,850] A new study created in memory with name: no-name-208e20cf-7f6a-4557-8b68-bf9586d400c0


--- Starting: LogisticRegression ---
Step 1: Tuning with 20 trials...


[I 2025-11-18 17:05:15,774] Trial 0 finished with value: 0.9109808117389869 and parameters: {'C': 0.0010971208178226328}. Best is trial 0 with value: 0.9109808117389869.
[I 2025-11-18 17:06:07,016] Trial 1 finished with value: 0.9111090902079442 and parameters: {'C': 6.657075959146137}. Best is trial 1 with value: 0.9111090902079442.
[I 2025-11-18 17:06:33,458] Trial 2 finished with value: 0.910691694174335 and parameters: {'C': 0.0003747263290509931}. Best is trial 1 with value: 0.9111090902079442.
[I 2025-11-18 17:07:23,826] Trial 3 finished with value: 0.9111090735419232 and parameters: {'C': 20.182912562987084}. Best is trial 1 with value: 0.9111090902079442.
[I 2025-11-18 17:08:13,686] Trial 4 finished with value: 0.9111146788104598 and parameters: {'C': 0.15322815819982766}. Best is trial 4 with value: 0.9111146788104598.
[I 2025-11-18 17:09:04,540] Trial 5 finished with value: 0.9111105390103631 and parameters: {'C': 0.6193074162620599}. Best is trial 4 with value: 0.91111467881

   ✅ Best Score: 0.911127
   ✅ Best Params: {'C': 0.014547150931384064}

Step 2: Generating predictions...

   ✅ Final OOF ROC AUC for LogisticRegression: 0.911126
--- Completed: LogisticRegression ---


460

In [ ]:
# ---  Create Submission for Logistic Regression ---

print("--- Creating submission file for Logistic Regression ---")

# 1. Define the model we want to submit
model_to_submit = 'LogisticRegression'

# 2. Get the test predictions for this model from our results dictionary
model_test_preds = test_preds_all[model_to_submit]

# 3. Get the original test IDs that we saved in Cell 2
if 'test_ids' in locals():
    print(f"   - Found test IDs. Shape: {test_ids.shape}")
    print(f"   - Found test predictions. Shape: {model_test_preds.shape}")

    # 4. Create the submission DataFrame with 'id' and the target column
    submission_df = pd.DataFrame({
        'id': test_ids,
        'loan_default': model_test_preds
    })

    # 5. Define the filename and save the CSV
    submission_filename = 'submission_LogisticRegression.csv'
    submission_df.to_csv(submission_filename, index=False)

    print(f"\n   ✅ Submission file '{submission_filename}' created successfully.")
    print("Top 5 rows of the submission file:")
    print(submission_df.head())

else:
    print("\n   ❌ ERROR: Could not find the 'test_ids' variable. Please re-run Cell 2 to load it.")

--- Creating submission file for Logistic Regression ---
   - Found test IDs. Shape: (254569,)
   - Found test predictions. Shape: (254569,)

   ✅ Submission file 'submission_LogisticRegression.csv' created successfully.
Top 5 rows of the submission file:
       id  loan_default
0  593994      0.763387
1  593995      0.929121
2  593996      0.046242
3  593997      0.760374
4  593998      0.893553


# **LIGHTGBM**

In [ ]:
# --- Prepare Data for Native Categorical Handling ---

print("--- Preparing data for high-performance training ---")

# Create copies to avoid modifying the original X and X_test
# We will use these 'native' versions for LightGBM and CatBoost
X_native = X.copy()
X_test_native = X_test.copy()

# Convert object columns to the 'category' dtype.
# This is the special format that LightGBM and CatBoost use for maximum speed.
for col in categorical_features:
    X_native[col] = X_native[col].astype('category')
    X_test_native[col] = X_test_native[col].astype('category')

print("Data types converted successfully.")
print("\nVerifying dtypes for X_native:")
# .info() is a great way to check that the conversion worked
X_native.info()

--- Preparing data for high-performance training ---
Data types converted successfully.

Verifying dtypes for X_native:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 593994 entries, 0 to 593993
Data columns (total 23 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   annual_income             593994 non-null  float64 
 1   debt_to_income_ratio      593994 non-null  float64 
 2   credit_score              593994 non-null  int64   
 3   loan_amount               593994 non-null  float64 
 4   interest_rate             593994 non-null  float64 
 5   gender                    593994 non-null  category
 6   marital_status            593994 non-null  category
 7   education_level           593994 non-null  category
 8   employment_status         593994 non-null  category
 9   loan_purpose              593994 non-null  category
 10  grade_subgrade            593994 non-null  category
 11  is_unemployed          

In [ ]:
# --- LightGBM - Part 1: Tuning ONLY ---

MODEL_NAME = 'LightGBM'
print(f"--- Starting: {MODEL_NAME} (Part 1: Tuning) ---")

# Initialize the tuner
tuner = ModelTuner(numerical_features, categorical_features, n_splits=N_SPLITS, seed=SEED)
n_trials = N_TRIALS_BOOSTING

print(f"Step 1: Tuning with {n_trials} trials using pre-converted 'category' data...")
print("This is the long-running step. Please be patient.")

# Run the tuning and store the results in memory
best_params, best_value = tuner.tune(MODEL_NAME, n_trials, X_native, y)

print("\n--- Tuning Complete! ---")
print(f"   ✅ Best Score found: {best_value:.6f}")
print(f"   ✅ Best Parameters found: {best_params}")
print("The results are now stored in memory. Proceed to Part 2.")


[I 2025-11-18 16:00:04,089] A new study created in memory with name: no-name-c3a0ff26-1b1a-4899-a38d-7b16b0a8074a


--- Starting: LightGBM (Part 1: Tuning) ---
Step 1: Tuning with 30 trials using pre-converted 'category' data...
This is the long-running step. Please be patient.


[I 2025-11-18 16:03:36,732] Trial 0 finished with value: 0.920728424230634 and parameters: {'n_estimators': 914, 'learning_rate': 0.020343431410788825, 'scale_pos_weight': 1.2262171784049345, 'num_leaves': 208}. Best is trial 0 with value: 0.920728424230634.
[I 2025-11-18 16:03:52,918] Trial 1 finished with value: 0.9105971056077884 and parameters: {'n_estimators': 1027, 'learning_rate': 0.06595828552116943, 'scale_pos_weight': 6.271754836131825, 'num_leaves': 38}. Best is trial 0 with value: 0.920728424230634.
[I 2025-11-18 16:04:31,576] Trial 2 finished with value: 0.915342991875446 and parameters: {'n_estimators': 974, 'learning_rate': 0.03513073603266558, 'scale_pos_weight': 4.934211992949362, 'num_leaves': 240}. Best is trial 0 with value: 0.920728424230634.
[I 2025-11-18 16:05:05,525] Trial 3 finished with value: 0.9130162938056756 and parameters: {'n_estimators': 1099, 'learning_rate': 0.03072645116978998, 'scale_pos_weight': 7.677778708368319, 'num_leaves': 208}. Best is trial 


--- Tuning Complete! ---
   ✅ Best Score found: 0.921071
   ✅ Best Parameters found: {'n_estimators': 893, 'learning_rate': 0.054914019153856616, 'scale_pos_weight': 2.1575155245000217, 'num_leaves': 62}
The results are now stored in memory. Proceed to Part 2.


In [ ]:
# --- LightGBM - Part 2: Save Results & Generate Predictions ---

print(f"--- Starting: {MODEL_NAME} (Part 2: Saving and Predicting) ---")

# --- FIX: Initialize dictionaries if they don't exist ---
# This makes this part of the process robust and self-contained.
if 'best_params_all' not in locals():
    print("Initializing results dictionaries...")
    best_params_all = {}
    oof_preds_all = {}
    test_preds_all = {}
# --- END OF FIX ---

# 1. Save the best parameters found in Part 1
best_params_all[MODEL_NAME] = best_params
print("   - Best parameters saved to dictionary.")

# 2. Generate OOF and Test predictions using the best parameters
print(f"   - Generating predictions with early stopping...")
model = tuner.models[MODEL_NAME]
model.set_params(**best_params)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_native, y)):
    print(f"     - Processing Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X_native.iloc[train_idx], X_native.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test_native)[:, 1] / N_SPLITS

# 3. Store the prediction results
oof_preds_all[MODEL_NAME] = oof_preds
test_preds_all[MODEL_NAME] = test_preds
final_oof_score = roc_auc_score(y, oof_preds)
print(f"\n   ✅ Final OOF ROC AUC for {MODEL_NAME}: {final_oof_score:.6f}")
print(f"--- Completed: {MODEL_NAME} ---")
gc.collect()

--- Starting: LightGBM (Part 2: Saving and Predicting) ---
   - Best parameters saved to dictionary.
   - Generating predictions with early stopping...
     - Processing Fold 1/5...
     - Processing Fold 2/5...
     - Processing Fold 3/5...
     - Processing Fold 4/5...
     - Processing Fold 5/5...

   ✅ Final OOF ROC AUC for LightGBM: 0.921071
--- Completed: LightGBM ---


671

In [ ]:
# --- Save All OOF and Test Predictions (Corrected with IDs) ---

print("--- Saving all collected predictions to CSV files ---")

# 1. Create a DataFrame for the Out-of-Fold (OOF) predictions
oof_df = pd.DataFrame(oof_preds_all)
oof_df = oof_df.add_prefix('oof_')
print(f"OOF DataFrame created. Shape: {oof_df.shape}")
print("OOF DataFrame columns:", oof_df.columns.tolist())

# 2. Create a DataFrame for the Test predictions
test_preds_df = pd.DataFrame(test_preds_all)
test_preds_df = test_preds_df.add_prefix('test_')
print(f"\nTest Predictions DataFrame created. Shape: {test_preds_df.shape}")

# --- THIS IS THE FIX ---
# 3. Add the original test IDs as the first column.
# The 'test_ids' variable was created in Cell 2.
test_preds_df.insert(0, 'id', test_ids)
print(f"   - 'id' column added to Test Predictions DataFrame.")
# --- END OF FIX ---

print("Test Predictions DataFrame columns:", test_preds_df.columns.tolist())

# 4. Save both DataFrames to CSV files
oof_df.to_csv('boosting_oof_preds.csv', index=False)
test_preds_df.to_csv('boosting_test_preds.csv', index=False)

print("\n✅ All predictions saved successfully to:")
print("   - boosting_oof_preds.csv")
print("   - boosting_test_preds.csv")

# Display the first few rows to verify the structure
print("\n--- OOF Predictions Head ---")
print(oof_df.head())
print("\n--- Test Predictions Head (with id) ---")
print(test_preds_df.head())


--- Saving all collected predictions to CSV files ---
OOF DataFrame created. Shape: (593994, 1)
OOF DataFrame columns: ['oof_LightGBM']

Test Predictions DataFrame created. Shape: (254569, 1)
   - 'id' column added to Test Predictions DataFrame.
Test Predictions DataFrame columns: ['id', 'test_LightGBM']

✅ All predictions saved successfully to:
   - boosting_oof_preds.csv
   - boosting_test_preds.csv

--- OOF Predictions Head ---
   oof_LightGBM
0      0.994027
1      0.709209
2      0.949981
3      0.948407
4      0.987196

--- Test Predictions Head (with id) ---
       id  test_LightGBM
0  593994       0.962743
1  593995       0.992429
2  593996       0.681136
3  593997       0.962114
4  593998       0.983710
